# yeast-1_vs_7: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('38_yeast-1_vs_7_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,16.330,11.429,40.000,6.667,100.000,53.333,51.413,73.317,13.879
smote,67.648,34.614,27.620,53.333,88.464,70.899,35.018,77.975,51.131
bsmote,68.812,36.226,27.766,53.333,90.211,71.772,40.503,79.843,52.519
ros,70.469,38.179,30.155,56.667,89.860,73.264,40.365,79.974,54.324
adasyn,66.631,31.087,22.793,51.667,86.135,68.901,32.108,76.479,48.859
casb,58.659,40.787,60.077,36.667,96.966,66.816,48.510,79.275,49.723
casb_relaxed,67.889,38.076,34.319,51.667,91.487,71.577,40.076,79.506,52.983
casb_edited,71.039,35.358,25.925,58.333,87.884,73.109,33.855,78.594,53.198
casb_ros,61.373,44.361,62.452,40.000,97.316,68.658,51.760,79.352,52.867


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,8.165,5.714,20.000,3.333,100.000,51.667,38.970,79.075,6.940
smote,74.745,34.147,23.229,68.333,84.157,76.245,29.623,77.283,54.446
bsmote,70.434,36.637,27.207,56.667,89.628,73.147,34.784,77.755,53.535
ros,74.885,37.768,26.655,66.667,87.070,76.868,34.835,79.579,56.326
adasyn,75.072,32.762,21.574,70.000,82.408,76.204,30.099,78.063,53.917
casb,71.328,36.665,26.980,60.000,88.699,74.350,42.022,78.673,53.996
casb_relaxed,72.363,34.497,24.113,63.333,86.249,74.791,35.203,78.560,53.430
casb_edited,73.554,32.297,21.496,66.667,83.098,74.883,27.093,76.569,52.925
casb_ros,70.470,36.637,27.486,58.333,88.819,73.576,41.995,79.491,53.553


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,27.795,20.159,46.667,13.333,99.416,56.375,42.750,83.863,23.977
smote,52.653,33.964,47.524,30.000,96.501,63.250,40.921,84.332,43.309
bsmote,51.599,34.986,52.357,28.333,97.315,62.824,39.950,83.578,43.293
ros,43.712,31.970,46.500,25.000,97.784,61.392,42.967,84.959,37.841
adasyn,49.294,29.772,40.190,26.667,96.501,61.584,34.787,82.443,39.533
casb,46.302,35.232,47.667,28.333,98.018,63.176,47.237,83.747,40.767
casb_relaxed,54.004,36.168,45.357,31.667,96.616,64.141,40.775,81.903,45.086
casb_edited,59.062,35.240,34.513,38.333,94.167,66.250,38.538,82.668,47.151
casb_ros,49.087,36.828,56.333,28.333,98.248,63.290,46.344,83.528,42.957


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,27.733,19.444,36.667,13.333,99.067,56.200,38.822,84.332,23.589
smote,69.796,31.447,21.686,60.000,85.082,72.541,38.074,84.472,50.622
bsmote,69.363,34.040,24.826,56.667,88.104,72.385,41.633,83.890,51.701
ros,70.851,32.429,22.391,61.667,84.844,73.255,40.072,84.471,51.640
adasyn,75.566,35.709,24.395,68.333,84.964,76.649,39.538,84.707,55.638
casb,67.520,33.416,24.755,53.333,88.342,70.838,38.719,84.351,50.468
casb_relaxed,67.813,32.463,23.634,55.000,87.294,71.147,38.461,84.238,50.138
casb_edited,72.998,34.715,23.997,65.000,85.661,75.330,39.786,84.258,53.857
casb_ros,68.571,34.591,25.737,55.000,88.338,71.669,39.070,84.408,51.581


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,20.006,14.187,35.833,9.167,99.621,54.394,42.988,80.147,17.096
smote,66.210,33.543,30.015,52.917,88.551,70.734,35.909,81.016,49.877
bsmote,65.052,35.472,33.039,48.750,91.314,70.032,39.217,81.267,50.262
ros,64.979,35.086,31.425,52.500,89.890,71.195,39.559,82.246,50.033
adasyn,66.641,32.333,27.238,54.167,87.502,70.834,34.133,80.423,49.487
casb,60.952,36.525,39.870,44.583,93.006,68.795,44.122,81.511,48.739
casb_relaxed,65.517,35.301,31.856,50.417,90.411,70.414,38.629,81.052,50.409
casb_edited,69.163,34.402,26.483,57.083,87.702,72.393,34.818,80.522,51.783
casb_ros,62.375,38.104,43.002,45.417,93.180,69.298,44.792,81.695,50.240


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.039,ros,70.469,0.569,WIN
1,f1,casb_ros,44.361,ros,38.179,6.183,WIN
2,precision,casb_ros,62.452,ros,30.155,32.298,WIN
3,recall,casb_edited,58.333,ros,56.667,1.667,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,73.554,adasyn,75.072,-1.517,LOSS
1,f1,casb,36.665,ros,37.768,-1.103,LOSS
2,precision,casb_ros,27.486,bsmote,27.207,0.278,WIN
3,recall,casb_edited,66.667,adasyn,70.000,-3.333,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,59.062,smote,52.653,6.410,WIN
1,f1,casb_ros,36.828,bsmote,34.986,1.842,WIN
2,precision,casb_ros,56.333,bsmote,52.357,3.976,WIN
3,recall,casb_edited,38.333,smote,30.000,8.333,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,69.163,adasyn,66.641,2.523,WIN
1,f1,casb_ros,38.104,bsmote,35.472,2.632,WIN
2,precision,casb_ros,43.002,bsmote,33.039,9.963,WIN
3,recall,casb_edited,57.083,adasyn,54.167,2.917,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.603697,0.920543,0.000000,None
1,42,1,svm,smote,0.50,0.396203,0.166667,0.166667,0.166667,0.941860,0.554264,0.338030,0.887597,0.281435,None
2,42,1,svm,bsmote,0.50,0.777593,0.444444,0.333333,0.666667,0.906977,0.786822,0.319328,0.903101,0.611019,None
3,42,1,svm,ros,0.50,0.690467,0.461538,0.428571,0.500000,0.953488,0.726744,0.377807,0.912791,0.576003,None
4,42,1,svm,adasyn,0.75,0.660338,0.300000,0.214286,0.500000,0.872093,0.686047,0.363942,0.852713,0.480169,None
5,42,1,svm,casb,0.50,0.573984,0.444444,0.666667,0.333333,0.988372,0.660853,0.580177,0.953488,0.509214,None
6,42,1,svm,casb_relaxed,0.75,0.767559,0.400000,0.285714,0.666667,0.883721,0.775194,0.291552,0.872093,0.583780,None
7,42,1,svm,casb_edited,0.50,0.767559,0.400000,0.285714,0.666667,0.883721,0.775194,0.376294,0.881783,0.583780,None
8,42,1,svm,casb_ros,0.50,0.573984,0.444444,0.666667,0.333333,0.988372,0.660853,0.736111,0.961240,0.509214,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.425162,0.945736,0.000000,None
